#### LIBRARY IMPORTS

In [1]:
import pandas as pd
import numpy as np
import glob # For file pattern matching for loading files
import os # Certain debugging operations
import joblib # To save scaler and encoder
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)

# Grid Search 
from itertools import product

# Neural Netowrk specifc imports
import copy # for deepcopy in save_results()

import torch
import torch.nn as nn # NN layers and loss functions
import torch.optim as optim # Optimization Algorithms
# Batching Data:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

#### CONFIGURATOINS

In [2]:
FEATURE_ROOT = "speaker_wise-eGeMAPs/functionals-unmerged_speakers/MP4_raw/"
OUTPUT_ROOT = "model_outputs/child-unmerged_gemaps/MP4_raw_grid_search/neural-network"
RANDOM_SEED = 42
THRESHOLD = 0.5
# TRAIN_RATIO = 0.8
# VAL_RATIO = 0.1
# TEST_RATIO = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PARAM_GRID = {
    "hidden1": [16, 32, 64, 128],
    "hidden2": [16, 32, 64],
    "dropout": [0.2, 0.3, 0.4, 0.5],
    "learning_rate": [1e-3, 5e-4],
    "weight_decay": [0, 1e-4]
}


CHILD_SPEAKER = {
    "speaker_functional_p5-s2.csv": ["SPEAKER_00"],
    "speaker_functional_p5-s7.csv": ["SPEAKER_01"],
    "speaker_functional_p5-s8.csv": ["SPEAKER_05"],
    "speaker_functional_p5-s10.csv": ["SPEAKER_01"],
    "speaker_functional_p5-s13.csv": ["SPEAKER_01"],

    "speaker_functional_p7-s5.csv": ["SPEAKER_01"],
    "speaker_functional_p7-s6.csv": ["SPEAKER_08","SPEAKER_05"],
    # "speaker_functional_p7-s7.csv": ["SPEAKER_02"], # Same as Kiwi
    "speaker_functional_p7-s8.csv": ["SPEAKER_00","SPEAKER_05"],
    "speaker_functional_p7-s16.csv": ["SPEAKER_00"],
    "speaker_functional_p7-s17.csv": ["SPEAKER_00","SPEAKER_02"],
    "speaker_functional_p7-s18.csv": ["SPEAKER_02"],
    "speaker_functional_p7-s29.csv": ["SPEAKER_00"],

    "speaker_functional_p9-s3-1.csv": ["SPEAKER_01"],
    "speaker_functional_p9-s3-2.csv": ["SPEAKER_04"],
    "speaker_functional_p9-s4.csv": ["SPEAKER_06"],
    "speaker_functional_p9-s9.csv": ["SPEAKER_03"],
    "speaker_functional_p9-s15.csv": ["SPEAKER_01"],

    "speaker_functional_p11-s2.csv": ["SPEAKER_02"],
    "speaker_functional_p11-s4.csv": ["SPEAKER_01"],
    "speaker_functional_p11-s8.csv": ["SPEAKER_03"],
    "speaker_functional_p11-s9.csv": ["SPEAKER_00"],
    "speaker_functional_p11-s11.csv": ["SPEAKER_05"],
    "speaker_functional_p11-s15.csv": ["SPEAKER_00"],
    # "speaker_functional_p11-s16-2.csv": ["SPEAKER_01"], # Same as Kiwi
    "speaker_functional_p11-s19.csv": ["SPEAKER_00"],
    "speaker_functional_p11-s22-2.csv": ["SPEAKER_02"],

    "speaker_functional_p12-s2-2.csv": ["SPEAKER_00"],
    # "speaker_functional_p12-s3.csv": ["SPEAKER_01"], # Same as Kiwi
    "speaker_functional_p12-s6.csv": ["SPEAKER_01"],
    # "speaker_functional_p12-s8.csv": ["SPEAKER_00"], # Same as Kiwi
    "speaker_functional_p12-s10.csv": ["SPEAKER_03"],

    "speaker_functional_p17-s2.csv": ["SPEAKER_01"],
    "speaker_functional_p17-s3.csv": ["SPEAKER_04"],
    "speaker_functional_p17-s5.csv": ["SPEAKER_04","SPEAKER_03","SPEAKER_05"],
    "speaker_functional_p17-s6.csv": ["SPEAKER_02"],

    "speaker_functional_p18-s3.csv": ["SPEAKER_00"],
    "speaker_functional_p18-s4.csv": ["SPEAKER_01"],
    # "speaker_functional_p18-s5.csv": ["SPEAKER_00"], # Same as Kiwi
    "speaker_functional_p18-s7.csv": ["SPEAKER_01"],
    "speaker_functional_p18-s8.csv": ["SPEAKER_00"],
    "speaker_functional_p18-s9.csv": ["SPEAKER_01"],
    "speaker_functional_p18-s10.csv": ["SPEAKER_06","SPEAKER_05"],
    "speaker_functional_p18-s11.csv": ["SPEAKER_00"],
    "speaker_functional_p18-s12.csv": ["SPEAKER_00"],
    "speaker_functional_p18-s13.csv": ["SPEAKER_01"],
    "speaker_functional_p18-s15.csv": ["SPEAKER_02","SPEAKER_01"],
    # "speaker_functional_p18-s17.csv": ["SPEAKER_01"], # Same as Kiwi
    "speaker_functional_p18-s18.csv": ["SPEAKER_00"],
    "speaker_functional_p18-s19.csv": ["SPEAKER_02"],
    "speaker_functional_p18-s20.csv": ["SPEAKER_01"],
}

# CHILD_AND_PARENT_SPEAKER = {
#     "speaker_functional_p5-s2.csv": ["SPEAKER_00","SPEAKER_02","SPEAKER_03"],
#     "speaker_functional_p5-s7.csv": ["SPEAKER_01","SPEAKER_02","SPEAKER_03"],
#     "speaker_functional_p5-s8.csv": ["SPEAKER_05","SPEAKER_00"],
#     "speaker_functional_p5-s10.csv": ["SPEAKER_01","SPEAKER_03"],
#     "speaker_functional_p5-s13.csv": ["SPEAKER_01","SPEAKER_04"],

#     "speaker_functional_p7-s5.csv": ["SPEAKER_01","SPEAKER_02"],
#     "speaker_functional_p7-s6.csv": ["SPEAKER_05", "SPEAKER_08", "SPEAKER_01", "SPEAKER_03"],
#     # "speaker_functional_p7-s7.csv": "SPEAKER_02", # Child and Kiwi not really differentiated
#     "speaker_functional_p7-s7.csv": ["SPEAKER_01"],
#     "speaker_functional_p7-s8.csv": ["SPEAKER_00",  "SPEAKER_05", "SPEAKER_03"],
#     "speaker_functional_p7-s16.csv": ["SPEAKER_00",  "SPEAKER_02"],
#     "speaker_functional_p7-s17.csv": ["SPEAKER_00", "SPEAKER_02"],
#     "speaker_functional_p7-s18.csv": ["SPEAKER_02",  "SPEAKER_01"],
#     # "speaker_functional_p7-s29.csv": "SPEAKER_00", # Child and Kiwi not really differentiated
#     "speaker_functional_p7-s29.csv": ["SPEAKER_04"], 

#     "speaker_functional_p9-s3-1.csv": ["SPEAKER_01", "SPEAKER_06", "SPEAKER_04", "SPEAKER_08"],
#     "speaker_functional_p9-s3-2.csv": ["SPEAKER_04", "SPEAKER_01"],
#     "speaker_functional_p9-s4.csv": ["SPEAKER_05",  "SPEAKER_04", "SPEAKER_06"],
#     "speaker_functional_p9-s9.csv": ["SPEAKER_03",  "SPEAKER_01", "SPEAKER_02"],
#     "speaker_functional_p9-s15.csv": ["SPEAKER_01",  "SPEAKER_02"],

#     "speaker_functional_p11-s2.csv": ["SPEAKER_02", "SPEAKER_05"],
#     "speaker_functional_p11-s4.csv": ["SPEAKER_01",  "SPEAKER_02"],
#     "speaker_functional_p11-s8.csv": ["SPEAKER_03", "SPEAKER_00"],
#     # "speaker_functional_p11-s9.csv": "SPEAKER_00", # Child and Kiwi not really differentiated
#     "speaker_functional_p11-s9.csv": ["SPEAKER_01"],
#     "speaker_functional_p11-s11.csv": ["SPEAKER_05", "SPEAKER_02", "SPEAKER_01"],
#     "speaker_functional_p11-s15.csv": ["SPEAKER_00"], # Both child and parent
#     # "speaker_functional_p11-s16-2.csv": "SPEAKER_01", # Child and Kiwi not really differentiated
#     "speaker_functional_p11-s16-2.csv": ["SPEAKER_02"],
#     "speaker_functional_p11-s19.csv": ["SPEAKER_02",  "SPEAKER_01"],
#     "speaker_functional_p11-s22-2.csv": ["SPEAKER_02","SPEAKER_00"],

#     "speaker_functional_p12-s2-2.csv": ["SPEAKER_00","SPEAKER_02"],
#     # "speaker_functional_p12-s3.csv": "SPEAKER_01", # Child, Parent and Kiwi not really differentiated
#     "speaker_functional_p12-s6.csv": ["SPEAKER_01","SPEAKER_02"],
#     # "speaker_functional_p12-s8.csv": "SPEAKER_00", # Child and Kiwi not really differentiated
#     "speaker_functional_p12-s8.csv": ["SPEAKER_01"],
#     "speaker_functional_p12-s10.csv": ["SPEAKER_01", "SPEAKER_03", "SPEAKER_02"],

#     "speaker_functional_p17-s2.csv": ["SPEAKER_01", "SPEAKER_02"],
#     "speaker_functional_p17-s3.csv": ["SPEAKER_04","SPEAKER_03"],
#     "speaker_functional_p17-s5.csv": ["SPEAKER_03",  "SPEAKER_04", "SPEAKER_05", "SPEAKER_00", "SPEAKER_01"],
#     "speaker_functional_p17-s6.csv": ["SPEAKER_00", "SPEAKER_01"],

#     "speaker_functional_p18-s3.csv": ["SPEAKER_00"],
#     "speaker_functional_p18-s4.csv": ["SPEAKER_01","SPEAKER_00"],
#     # "speaker_functional_p18-s5.csv": "SPEAKER_00", # Child and Kiwi not really differentiated
#     "speaker_functional_p18-s5.csv": ["SPEAKER_01"],
#     "speaker_functional_p18-s7.csv": ["SPEAKER_01"],
#     "speaker_functional_p18-s8.csv": ["SPEAKER_01",  "SPEAKER_00", "SPEAKER_03"],
#     "speaker_functional_p18-s9.csv": ["SPEAKER_01",  "SPEAKER_00"],
#     "speaker_functional_p18-s10.csv": [["SPEAKER_06",]],
#     "speaker_functional_p18-s11.csv": ["SPEAKER_00"],
#     "speaker_functional_p18-s12.csv": ["SPEAKER_00",  "SPEAKER_02"],
#     "speaker_functional_p18-s13.csv": ["SPEAKER_01"],
#     "speaker_functional_p18-s15.csv": ["SPEAKER_01",  "SPEAKER_02"],
#     # "speaker_functional_p18-s17.csv": "SPEAKER_01", #Child, Parent and Kiwi not really differentiated
#     "speaker_functional_p18-s18.csv": ["SPEAKER_00"],
#     "speaker_functional_p18-s19.csv": ["SPEAKER_02",  "SPEAKER_01"],
#     "speaker_functional_p18-s20.csv": ["SPEAKER_01"],
# }

# PARENT_SPEAKER = {
#     "speaker_functional_p5-s2.csv": ["SPEAKER_02","SPEAKER_03"],
#     "speaker_functional_p5-s7.csv": ["SPEAKER_02","SPEAKER_03"],
#     "speaker_functional_p5-s8.csv": ["SPEAKER_00"],
#     "speaker_functional_p5-s10.csv": ["SPEAKER_03"],
#     "speaker_functional_p5-s13.csv": ["SPEAKER_04"],

#     "speaker_functional_p7-s5.csv": ["SPEAKER_02"],
#     "speaker_functional_p7-s6.csv": ["SPEAKER_01", "SPEAKER_03"],
#     # "speaker_functional_p7-s7.csv": "SPEAKER_02", # Child and Kiwi not really differentiated
#     "speaker_functional_p7-s7.csv": ["SPEAKER_01"],
#     "speaker_functional_p7-s8.csv": ["SPEAKER_03"],
#     "speaker_functional_p7-s16.csv": ["SPEAKER_02"],
#     "speaker_functional_p7-s17.csv": ["SPEAKER_02"],
#     "speaker_functional_p7-s18.csv": ["SPEAKER_01"],
#     # "speaker_functional_p7-s29.csv": "SPEAKER_00", # Child and Kiwi not really differentiated
#     "speaker_functional_p7-s29.csv": ["SPEAKER_04"], 

#     "speaker_functional_p9-s3-1.csv": ["SPEAKER_06", "SPEAKER_04", "SPEAKER_08"],
#     "speaker_functional_p9-s3-2.csv": ["SPEAKER_01"],
#     "speaker_functional_p9-s4.csv": ["SPEAKER_04", "SPEAKER_06"],
#     "speaker_functional_p9-s9.csv": ["SPEAKER_01", "SPEAKER_02"],
#     "speaker_functional_p9-s15.csv": ["SPEAKER_02"],

#     "speaker_functional_p11-s2.csv": ["SPEAKER_05"],
#     "speaker_functional_p11-s4.csv": ["SPEAKER_02"],
#     "speaker_functional_p11-s8.csv": ["SPEAKER_00"],
#     # "speaker_functional_p11-s9.csv": "SPEAKER_00", # Child and Kiwi not really differentiated
#     "speaker_functional_p11-s9.csv": ["SPEAKER_01"],
#     "speaker_functional_p11-s11.csv": ["SPEAKER_02", "SPEAKER_01"],
#     # "speaker_functional_p11-s16-2.csv": "SPEAKER_01", # Child and Kiwi not really differentiated
#     "speaker_functional_p11-s16-2.csv": ["SPEAKER_02"],
#     "speaker_functional_p11-s19.csv": ["SPEAKER_01"],
#     "speaker_functional_p11-s22-2.csv": ["SPEAKER_00"],

#     "speaker_functional_p12-s2-2.csv": ["SPEAKER_02"],
#     # "speaker_functional_p12-s3.csv": "SPEAKER_01", # Child, Parent and Kiwi not really differentiated
#     "speaker_functional_p12-s6.csv": ["SPEAKER_01","SPEAKER_02"],
#     # "speaker_functional_p12-s8.csv": "SPEAKER_00", # Child and Kiwi not really differentiated
#     "speaker_functional_p12-s8.csv": ["SPEAKER_01"],
#     "speaker_functional_p12-s10.csv": ["SPEAKER_03", "SPEAKER_02"],

#     "speaker_functional_p17-s2.csv": ["SPEAKER_01", "SPEAKER_02"],
#     "speaker_functional_p17-s3.csv": ["SPEAKER_03"],
#     "speaker_functional_p17-s5.csv": ["SPEAKER_00", "SPEAKER_01"],
#     "speaker_functional_p17-s6.csv": ["SPEAKER_00", "SPEAKER_01"],

#     "speaker_functional_p18-s3.csv": ["SPEAKER_00"],
#     "speaker_functional_p18-s4.csv": ["SPEAKER_00"],
#     # "speaker_functional_p18-s5.csv": "SPEAKER_00", # Child and Kiwi not really differentiated
#     "speaker_functional_p18-s5.csv": ["SPEAKER_01"],
#     # "speaker_functional_p18-s7.csv": ["SPEAKER_01"], # N/A
#     "speaker_functional_p18-s8.csv": ["SPEAKER_00", "SPEAKER_03"],
#     "speaker_functional_p18-s9.csv": ["SPEAKER_00"],
#     "speaker_functional_p18-s10.csv": ["SPEAKER_03"],
#     # "speaker_functional_p18-s11.csv": ["SPEAKER_00"], #N/A
#     "speaker_functional_p18-s12.csv": ["SPEAKER_02"],
#     # "speaker_functional_p18-s13.csv": ["SPEAKER_01"],
#     # "speaker_functional_p18-s15.csv": ["SPEAKER_01",  "SPEAKER_02"],
#     # "speaker_functional_p18-s17.csv": "SPEAKER_01", #Child, Parent and Kiwi not really differentiated
#     # "speaker_functional_p18-s18.csv": ["SPEAKER_00"],
#     "speaker_functional_p18-s19.csv": ["SPEAKER_01"],
#     "speaker_functional_p18-s20.csv": ["SPEAKER_01"],
# }



#### FUNCTIONS

In [3]:
# Early stopping
PATIENCE = 10 # Stop after 10 epochs of no improvement in validation loss

In [ ]:
def load_participant_data(participant_folder):
    # Load all CSV files for a given participant folder in a sorted fashion
    csv_files = [
        f for f in sorted(glob.glob(f"{FEATURE_ROOT}/{participant_folder}/*.csv"))
        if os.path.basename(f) in CHILD_SPEAKER
    ] # For multiple files

    #  Function to filter the child speaker from single csv file
    def load_single_speaker(csv_path):
        df = pd.read_csv(csv_path)    
        filename = os.path.basename(csv_path)

        # Error handle
        if filename not in CHILD_SPEAKER:
            raise ValueError(
                f"No child speaker mapping for {filename}"
            )
        
        target_speakers = CHILD_SPEAKER[filename]
        
        # df = df[True] where it is True for child speaker
        df = df[df["speaker"].isin(target_speakers)]

        return df

    ### DEBUG STATEMENT   
    print(f"{participant_folder}: {len(csv_files)} CSV files")
    ###

    # Load every session for this participant
    full_df = pd.concat(
        [
            # Load child speaker only from file f
            load_single_speaker(f)
            for f in csv_files
        ],
        ignore_index=True
    )
     
    ### DEBUG STATEMENT
    print(f"Total child+parent utterances: {len(full_df)}")
    ### 

    # Remove unnecessary columns from eGeMAPs table
    drop_cols = [
        "participant",
        "session",
        "engagement_id",
        "clip_id",
        "speaker",

        "engagement_start_time",
        "engagement_end_time",

        "speaker_start_time",
        "speaker_end_time",

        "num_segments",
        "speech_duration",
        "merged_duration",
    ]

    full_df = full_df.drop(
        columns=[c for c in drop_cols if c in full_df.columns]
    )

    print(full_df.columns)

    # Separate Features and Labels
    X = full_df.drop(columns=["label"])
    y = full_df["label"]

    # We reset the index below because we'ev filtered non-child speaker rows
    return (
        X.reset_index(drop=True),
        y.reset_index(drop=True)
    )

In [5]:
def preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test):
    
    # Label Encoding
    encoder = LabelEncoder()
    y_train = encoder.fit_transform(y_train)
    y_val = encoder.transform(y_val)
    y_test = encoder.transform(y_test)

    ### DEBUG STATEMENT
    print(encoder.classes_)
    print(np.unique(y_train))
    print(np.unique(y_val))
    print(np.unique(y_test))
    ###
    
    # Feature Scaling
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train) # Learn the Meand and std and transform the data
    X_val = scaler.transform(X_val) # Transform the data using the learned mean and std
    X_test = scaler.transform(X_test) # Transform the data using the learned mean and std

    return (
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        scaler,
        encoder
    )  

In [6]:
class NeuralNetwork(nn.Module):

    # Define the architecture of the neural network ; Constructor
    def __init__(
        self,
        input_dim,
        hidden1,
        hidden2,
        dropout
    ):

        super().__init__() # Initialize the parent class (nn.Module) first, then inherit functionalities

        self.network = nn.Sequential(
            # First Hidden Layer 88 -> 128 nueruons
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Second Hidden Layer 128 -> 64 nueruons
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(dropout),

            # Output Layer 64 -> 1 nuerons
            nn.Linear(hidden2,1)

        )
    
    # Forward pass through the network; Automatically called  when calling model; Then return the model predictions
    def forward(self, x):
        return self.network(x)


# Build the model, then move it to GPU/ CPU and print the model architecture
def build_model(
    input_dim,
    hidden1,
    hidden2,
    dropout
):

    model = NeuralNetwork(
        input_dim,
        hidden1,
        hidden2,
        dropout
    )

    return model.to(DEVICE)

In [7]:
def train_model(
    model,
    train_loader,
    val_loader,
    learning_rate,
    weight_decay
):
    # BCEWithLogitsLoss is good for our case of binary classification
    # It performs sigmoid + Binary Cross Entropy Loss 
    criterion = nn.BCEWithLogitsLoss()

    # Adam optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    # Initialize best loss to be infinity and patience counter to 0
    best_loss = float("inf")
    patience_counter = 0

    # Store training and validation loss for each epoch (perhaps for plotting later)
    history = {
        "train_loss": [],
        "val_loss": []
    }

    # Epoch Loop; Max is 100, but may stop earlier due to early stopping
    for epoch in range(100):
        
        # TRAINING STEPS:
        # Activate training mode (Dropout layers and gradient computation)
        model.train()
        train_loss = 0

        # Train Batch-wise (32)
        for X_batch, y_batch in train_loader:
            
            # Move batch to GPU/ CPU
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            # Reset the gradients before backpropagation
            optimizer.zero_grad()

            # Foeward pass prediction
            outputs = model(X_batch)

            # Compute loss
            loss = criterion(outputs, y_batch)

            # Backpropagation 
            # Computes gradients
            loss.backward()
            # Update weights using optimizer
            optimizer.step()
            # Accumute training loss
            train_loss += loss.item() # loss is a tensor

        # Average traingin loss per batch
        train_loss /= len(train_loader)

        # VALIDATION STEPS
        # Turn off training mode( No Dropout layers and gradient computation)
        model.eval()
        val_loss = 0

        # Disable gradient computation
        with torch.no_grad():
            # Validate Batch-wise (32)
            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
 
                # Calls the forward pass of the model to get predictions
                outputs = model(X_batch)
                # Compute validation loss; criterion is the loss function defined in above 
                loss = criterion(outputs, y_batch)
                #Accumulate validation loss
                val_loss += loss.item()

        val_loss /= len(val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1} "
            f"Train={train_loss:.4f} "
            f"Val={val_loss:.4f}"
        )

        # If validation loss improves, save the model weights and reset patience counter
        if val_loss < best_loss:
            best_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0

        # If validation loss does not improve, increment patience counter and check for early stopping
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("Early stopping")
                # Exit training loop
                break

    # Restore the best model weights after training is complete
    model.load_state_dict(best_weights)

    return history

In [8]:
def evaluate_model(
        model,
        test_loader,
        encoder
):
    # Turn off training mode( No Dropout layers and gradient computation)
    model.eval()

    probabilities = [] # Store the predicted probabilities for the positive class (engaged)
    predictions = [] # Store the predicted class labels (0 or 1)
    actual = [] # Store the actual class labels (0 or 1)

    # Disable gradient computation
    with torch.no_grad():
        # Exaluate Batch-wise (32)
        for X_batch, y_batch in test_loader:
            
            X_batch = X_batch.to(DEVICE)
            # Forward pass prediction
            outputs = model(X_batch)
            # Apply sigmoid to get probabilities (0 to 1)
            probs = torch.sigmoid(outputs)
            # Convert probabilities to binary predictions (0 or 1) using a threshold of 0.5
            preds = (probs >= THRESHOLD).float()

            # NumPy cannotread GPU tensors, so we need to move them to CPU and convert to NumPy arrays before storing them in lists
            probabilities.extend(probs.cpu().numpy().flatten())
            predictions.extend(preds.cpu().numpy().flatten())
            actual.extend(y_batch.numpy().flatten())

    # Convert lists to NumPy arrays and ensure they are of integer type
    probabilities = np.array(probabilities).astype(float)
    predictions = np.array(predictions).astype(int)
    actual = np.array(actual).astype(int)

    # DEBUGGING STATEMENTS
    # How many samples are actually there from each class
    print("\nActual class counts:")
    print(pd.Series(actual).value_counts()) 
    # How many samples were predicted from each class
    print("\nPredicted class counts:")
    print(pd.Series(predictions).value_counts())
    # Print first 20 probabilty values
    print("\nPredicted probabilities:")
    print(probabilities[:20])

    print(f"\nMinimum probability : {probabilities.min():.3f}")
    print(f"Maximum probability : {probabilities.max():.3f}")
    print(f"Average probability : {probabilities.mean():.3f}")

    # actual contains the true labels; filter the engaged and disengaged probabilities based on the actual labels
    engaged_probs = probabilities[actual == 1]
    disengaged_probs = probabilities[actual == 0]

    print("\nProbability Statistics")
    print("----------------------")
    print(f"Engaged mean      : {engaged_probs.mean():.3f}")
    print(f"Engaged std       : {engaged_probs.std():.3f}")
    print(f"Disengaged mean   : {disengaged_probs.mean():.3f}")
    print(f"Disengaged std    : {disengaged_probs.std():.3f}")

    # plt.figure(figsize=(6,4))
    # plt.hist(engaged_probs, bins=15, alpha=0.6, label="Engaged")
    # plt.hist(disengaged_probs, bins=15, alpha=0.6, label="Disengaged")
    # plt.xlabel("Predicted Probability")
    # plt.ylabel("Count")
    # plt.legend()
    # plt.show()

    # Confusion Matrix
    cm = confusion_matrix(actual, predictions)

    # Compute metrics
    metrics_dictionary = {
        "Accuracy": accuracy_score(actual, predictions),
        "Precision": precision_score(actual, predictions),
        "Recall": recall_score(actual, predictions),
        "F1 Score": f1_score(actual, predictions),
        "AUROC": roc_auc_score(actual, probabilities)
    }
    print(f"AUROC: {metrics_dictionary['AUROC']:.4f}")
    
    report = classification_report(
        actual,
        predictions,
        target_names=encoder.classes_,
        output_dict=True # Return the report as a dictionary instead of a string to make csv
    )

    print(classification_report(
        actual,
        predictions,
        target_names=encoder.classes_
    ))

    return {
        "metrics": metrics_dictionary,
        "predictions": predictions,
        "probabilities": probabilities,
        "actual": actual,
        "confusion_matrix": cm,
        "classification_report": report
    }

In [9]:
''' The following are saved:
    Model weights (model.pt)
    Scaler (scaler.pkl)
    Encoder (encoder.pkl)
    Training History (history.csv)
    Metrics (metrics.csv)
    Confusion Matrix (confusion_matrix.csv)
    Classification Report (classification_report.csv) '''
    
def save_results(
        participant,
        model,
        scaler,
        encoder,
        history,
        evaluation
    ):
    
    metrics = evaluation["metrics"]
    predictions = evaluation["predictions"]
    probabilities = evaluation["probabilities"]
    actual = evaluation["actual"]
    cm = evaluation["confusion_matrix"]
    report = evaluation["classification_report"]
    
    # Save Model
    participant_output = Path(OUTPUT_ROOT, participant)
    os.makedirs(participant_output, exist_ok=True)
    torch.save(model.state_dict(),participant_output / "neural-network.pt")
    
    # Save Scaler
    joblib.dump(scaler,Path(participant_output, "scaler.pkl"))
    
    # Save Encoder
    joblib.dump(encoder,Path(participant_output, "encoder.pkl"))
    
    # Save History
    history_df = pd.DataFrame(history)
    history_df.to_csv(Path(participant_output, "history.csv"),index=False)
    
    # Save Metrics
    metrics_df = pd.DataFrame([metrics])

    metrics_df.to_csv(Path(participant_output, "metrics.csv"),index=False)
        
    # Save Confusion Matrix
    cm_df = pd.DataFrame(cm,index=encoder.classes_,columns=encoder.classes_)
    cm_df.to_csv(Path(participant_output, "confusion_matrix.csv"))
    
    # Save classification report
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(Path(participant_output, "classification_report.csv"))
    
    # Save predictions
    prediction_df = pd.DataFrame({
        "Actual": actual,
        "Prediction": predictions,
        "Probability": probabilities
    })

    prediction_df.to_csv(Path(participant_output, "predictions.csv"),index=False)

#### RUN MODEL

In [10]:
dataset_statistics = [] # Store statistics for each participant
participant_statistics = [] # Store statistics for each participant
summary_results = [] # Store average metircs for each participant
participants = sorted(os.listdir(FEATURE_ROOT))
all_fold_results = [] # Store metrics for each fold of each participant

# Loop through each participant and train a model for each participant
for participant in participants:

    print(f"Training {participant}")
    # Load ALL data for this participant; X= GeMAPs features, y= labels (engaged/disengaged)
    X, y = load_participant_data(participant)

    participant_statistics.append({
        "Participant": participant,
        "Total Samples": len(y),
        "Engaged": (y == "engaged").sum(),
        "Disengaged": (y == "disengaged").sum(),
        "Sessions": X["session"].nunique() if "session" in X.columns else np.nan,
        "Speakers": X["speaker"].nunique() if "speaker" in X.columns else np.nan
    })

    grid = list(product(
        PARAM_GRID["hidden1"],
        PARAM_GRID["hidden2"],
        PARAM_GRID["dropout"],
        PARAM_GRID["learning_rate"],
        PARAM_GRID["weight_decay"]
    ))

    # Create the 5 folds once and reuse them throughout the grid search
    skf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_SEED
    )

    fold_splits = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]

        X_train, X_val, y_train, y_val = train_test_split(
            X_train,
            y_train,
            test_size=0.20,
            stratify=y_train,
            random_state=RANDOM_SEED
        )

        dataset_statistics.append({

            "Participant": participant,
            "Fold": fold,

            "Training Samples": len(y_train),
            "Training Engaged": (y_train == "engaged").sum(),
            "Training Disengaged": (y_train == "disengaged").sum(),

            "Validation Samples": len(y_val),
            "Validation Engaged": (y_val == "engaged").sum(),
            "Validation Disengaged": (y_val == "disengaged").sum(),

            "Testing Samples": len(y_test),
            "Testing Engaged": (y_test == "engaged").sum(),
            "Testing Disengaged": (y_test == "disengaged").sum(),
        })

        fold_splits.append(
            (
                fold,
                X_train.copy(),
                X_val.copy(),
                X_test.copy(),
                y_train.copy(),
                y_val.copy(),
                y_test.copy(),
            )
        )

    best_score = -1
    best_params = None
    best_results = None
    participant_grid_results = []

    for hidden1, hidden2, dropout, lr, wd in grid:
        participant_metrics = []

        print(
            f"\nTesting:"
            f" H1={hidden1}"
            f" H2={hidden2}"
            f" Dropout={dropout}"
            f" LR={lr}"
            f" WD={wd}"
        )
            
        for (
            fold,
            X_train,
            X_val,
            X_test,
            y_train,
            y_val,
            y_test,
        ) in fold_splits:

            # DEBUGGING STATEMENTS
            print("\nTraining class distribution:")
            print(y_train.value_counts())

            print("\nValidation class distribution:")
            print(y_val.value_counts())

            print("\nTesting class distribution:")
            print(y_test.value_counts())
            ###

            # Preprocess data
            X_train, X_val, X_test, y_train, y_val, y_test, scaler, encoder = preprocess_data(X_train, X_val, X_test, y_train, y_val, y_test)

            # Convert NumPy arrays into PyTorch tensors.
            X_train_tensor = torch.FloatTensor(X_train)
            X_val_tensor = torch.FloatTensor(X_val)
            X_test_tensor = torch.FloatTensor(X_test)

            # Shape of y_train, y_val, y_test is (N,), but we need (N,1) for BCEWithLogitsLoss
            y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
            y_val_tensor = torch.FloatTensor(y_val).unsqueeze(1)
            y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

            # Create TensorDatasets; Join features with corresponding labels
            train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
            val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
            test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

            # Create DataLoaders; Batch the data (32) and randomly shuffle
            # FOr trainging dataset
            train_loader = DataLoader(
                train_dataset,
                batch_size=32,
                shuffle=True # Shuffle training data for better generalization
            )
            # FOr validation dataset
            val_loader = DataLoader(
                val_dataset,
                batch_size=32,
                shuffle=False
            )
            # FOr testing dataset
            test_loader = DataLoader(
                test_dataset,
                batch_size=32,
                shuffle=False
            )

            # Build model
            model = build_model(
                X_train.shape[1],
                hidden1,
                hidden2,
                dropout
            )

            # Train model and return training history
            history = train_model(
                model,
                train_loader,
                val_loader,
                lr,
                wd
            )

            # Evaluate model and return evaluation metrics
            evaluation = evaluate_model(model, test_loader, encoder)

            # Append metrics for this participant to the summary results
            participant_metrics.append(evaluation["metrics"])

            # Append fold results to all_fold_results
            all_fold_results.append({
                "Participant": participant,
                "Fold": fold,
                "Hidden1": hidden1,
                "Hidden2": hidden2,
                "Dropout": dropout,
                "LearningRate": lr,
                "WeightDecay": wd,
                # evaluation["metrics"] is a dictionary containing the metrics for this fold, ** unpacks the dictionary 
                **evaluation["metrics"]
            })

            # Save results
            # save_results(
            #     f"{participant}/fold_{fold}",
            #     model,
            #     scaler,
            #     encoder,
            #     history,
            #     evaluation
            # )



        metrics_df = pd.DataFrame(participant_metrics)

        mean_f1 = metrics_df["F1 Score"].mean()

        participant_grid_results.append({

            "Hidden1": hidden1,
            "Hidden2": hidden2,
            "Dropout": dropout,
            "LearningRate": lr,
            "WeightDecay": wd,

            "Accuracy": metrics_df["Accuracy"].mean(),
            "Precision": metrics_df["Precision"].mean(),
            "Recall": metrics_df["Recall"].mean(),
            "F1": mean_f1,
            "AUROC": metrics_df["AUROC"].mean()
        })

        if mean_f1 > best_score:

            best_score = mean_f1

            best_params = {
                "Hidden1": hidden1,
                "Hidden2": hidden2,
                "Dropout": dropout,
                "LearningRate": lr,
                "WeightDecay": wd
            }

            best_results = metrics_df.copy()
    
    summary_results.append({
        "Participant": participant,

        "Best Hidden1": best_params["Hidden1"],
        "Best Hidden2": best_params["Hidden2"],
        "Best Dropout": best_params["Dropout"],
        "Best LearningRate": best_params["LearningRate"],
        "Best WeightDecay": best_params["WeightDecay"],

        "Accuracy Mean": best_results["Accuracy"].mean(),
        "Accuracy Std": best_results["Accuracy"].std(),

        "Precision Mean": best_results["Precision"].mean(),
        "Precision Std": best_results["Precision"].std(),

        "Recall Mean": best_results["Recall"].mean(),
        "Recall Std": best_results["Recall"].std(),

        "F1 Mean": best_results["F1 Score"].mean(),
        "F1 Std": best_results["F1 Score"].std(),

        "AUROC Mean": best_results["AUROC"].mean(),
        "AUROC Std": best_results["AUROC"].std(),
    })  

    participant_output = Path(OUTPUT_ROOT, participant)
    participant_output.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(participant_grid_results).to_csv(
        participant_output / "grid_search_results.csv",
        index=False
    )

pd.DataFrame(all_fold_results).to_csv(
        "fold_results.csv",
        index=False
    )

# Save summary results as CSV
summary_df = pd.DataFrame(summary_results)
summary_df.to_csv(
    Path(OUTPUT_ROOT,"summary_results.csv"),
    index=False
)

pd.DataFrame(dataset_statistics).to_csv(
    Path(OUTPUT_ROOT, "dataset_statistics.csv"),
    index=False
)

pd.DataFrame(participant_statistics).to_csv(
    Path(OUTPUT_ROOT, "participant_statistics.csv"),
    index=False
)

Training p11


NameError: name 'PARENT_SPEAKER' is not defined